In [ ]:
# =============================================================
# ONE-TIME SETUP — only run this cell once to install GPU PyTorch
# After it finishes, restart the kernel before continuing.
# =============================================================
import sys, subprocess

def install_gpu_torch():
    print('Removing old CPU build...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall',
                           'torch', 'torchvision', 'torchaudio', '-y'])
    print('Downloading CUDA 12.1 build (~2.5 GB)...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'torch', 'torchvision', 'torchaudio',
        '--index-url', 'https://download.pytorch.org/whl/cu121'
    ])
    print('Done! Restart the kernel now before running anything else.')

install_gpu_torch()


In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU ready: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: running on CPU — inference will be slow.')


In [ ]:
# All imports in one place
import platform
import signal
import gc
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

# FIX: guard SIGALRM patch properly — only apply on Windows when missing
if platform.system() == 'Windows' and not hasattr(signal, 'SIGALRM'):
    signal.SIGALRM = signal.SIGTERM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

model_name = 'InstaDeepAI/nucleotide-transformer-v2-500m-multi-species'

print('Loading tokenizer and model...')
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True).to(device).half()
model.eval()
print('Ready.')


In [ ]:
# Paste your sequence here
full_sequence = """TTCCAGGACTGCAGAACTGGCCCAGACCTCTGTATTGGAAAGGTCTTTATGGACCAGGGAGTCCGGTGTCTTTTTTACGGGGGACCCCTG
GGCTGCGAGTTGCACAGTCCAATTCGCTGTTGTTAGGGCCTCAGTTTCCCAAAAGGCACAGGGACGGGGGGAGGGTGGCGGCTCGATGGG
GGAGCCGCCTCCAGGGGGCCCCCCCGCCCTGTGCCCACGGCGCGGCCCCTTTAAGAGGCCCGCCTGGCTCCGTCATCCGCGCCGCGGCCA
CCTCCCCCCGGCCCTCCCCTTCCTGCGGCGCAGAGTGCGGGCCGGGCGGGAGTGCGGCGAGAGCCGGCTGGCTGAGCTTAGCGTCCGAGG
AGGCGGCGGCGGCGGCGGCGGCACGGCGGCGGCGGGGCTGTGGGGCGGTGCGGAAGCGAGAGGCGAGGAGCGCGCGGGCCGTGGCCAGAG
TCTGGCGGCGGCCTGGCGGAGCGGAGAGCAGCGCCCGCGCCTCGCCGTGCGGAGGAGCCCCGCACACAATAGCGGCGCGCGCAGCCCGCG
CCCTTCCCCCCGGCGCGCCCCGCCCCGCGCGCCGAGCGCCCCGCTCCGCCTCACCTGCCACCAGGGAGTGGGCGGGCATTGTTCGCCGCC
GCCGCCGCCGCGCGGGCCATGGGGGCCGCCCGGCGCCCGGGGCCGGGCTGGCGAGGCGCCGCGCCGCCGCTGAGACGGGCCCCGCGCGCA
GCCCGGCGGCGCAGGTAAGGCCGGCCGCGCCATGGTGGACCCGGTGGGCTTCGCGGAGGCGTGGAAGGCGCAGTTCCCGGACTCAGAGCC
CCCGCGCATGGAGCTGCGCTCAGTGGGCGACATCGAGCAGGAGCTGGAGCGCTGCAAGGCCTCCATTCGGCGCCTGGAGCAGGAGGTGAA
CCAGGAGCGCTTCCGCATGATCTACCTGCAGACGTTGCTGGCCAAGGAAAAGAAGAGCTATGACCGGCAGCGATGGGGCTTCCGGCGCGC
GGCGCAGGCCCCCGACGGCGCCTCCGAGCCCCGAGCGTCCGCGTCGCGCCCGCAGCCAGCGCCCGCCGACGGAGCCGACCCGCCGCCCGC
CGAGGAGCCCGAGGCCCGGCCCGACGGCGAGGGTTCTCCGGGTAAGGCCAGGCCCGGGACCGCCCGCAGGCCCGGGGCAGCCGCGTCGGG
GGAACGGGACGACCGGGGACCCCCCGCCAGCGTGGCGGCGCTCAGGTCCAACTTCGAGCGGATCCGCAAGGGCCATGGCCAGCCCGGGGC
GGACGCCGAGAAGCCCTTCTACGTGAACGTCGAGTTTCACCACGAGCGCGGCCTGGTGAAGGTCAACGACAAAGAGGTGTCGGACCGCAT
CAGCTCCCTGGGCAGCCAGGCCATGCAGATGGAGCGCAAAAAGTCCCAGCACGGCGCGGGCTCGAGCGTGGGGGATGCATCCAGGCCCCC
TTACCGGGGACGCTCCTCGGAGAGCAGCTGCGGCGTCGACGGCGACTACGAGGACGCCGAGTTGAACCCCCGCTTCCTGAAGGACAACCT
GATCGACGCCAATGGCGGTAGCAGGCCCCCTTGGCCGCCCCTGGAGTACCAGCCCTACCAGAGCATCTACGTCGGGGGCATGATGGAAGG
GGAGGGCAAGGGCCCGCTCCTGCGCAGCCAGAGCACCTCTGAGCAGGAGAAGCGCCTTACCTGGCCCCGCAGGTCCTACTCCCCCCGGAG
TTTTGAGGATTGCGGAGGCGGCTATACCCCGGACTGCAGCTCCAATGAGAACCTCACCTCCAGCGAGGAGGACTTCTCCTCTGGCCAGTC
CAGCCGCGTGTCCCCAAGCCCCACCACCTACCGCATGTTCCGGGACAAAAGCCGCTCTCCCTCGCAGAACTCGCAACAGTCCTTCGACAG
CAGCAGTCCCCCCACGCCGCAGTGCCATAAGCGGCACCGGCACTGCCCGGTTGTCGTGTCCGAGGCCACCATCGTGGGCGTCCGCAAGAC
CGGGCAGATCTGGCCCAACGATGGCGAGGGCGCCTTCCATGGAGACGCAGATGGCTCGTTCGGAACACCACCTGGATACGGCTGCGCTGC
AGACCGGGCAGAGGAGCAGCGCCGGCACCAAGATGGGCTGCCCTACATTGATGACTCGCCCTCCTCATCGCCCCACCTCAGCAGCAAGGG
CAGGGGCAGCCGGGATGCGCTGGTCTCGGGAGCCCTGGAGTCCACTAAAGCGAGTGAGCTGGACTTGGAAAAGGGCTTGGAGATGAGAAA
ATGGGTCCTGTCGGGAATCCTGGCTAGCGAGGAGACTTACCTGAGCCACCTGGAGGCACTGCTGCTGCCCATGAAGCCTTTGAAAGCCGC
TGCCACCACCTCTCAGCCGGTGCTGACGAGTCAGCAGATCGAGACCATCTTCTTCAAAGTGCCTGAGCTCTACGAGATCCACAAGGAGTT
CTATGATGGGCTCTTCCCCCGCGTGCAGCAGTGGAGCCACCAGCAGCGGGTGGGCGACCTCTTCCAGAAGCTGGCCAGCCAGCTGGGTGT
GTACCGGGCCTTCGTGGACAACTACGGAGTTGCCATGGAAATGGCTGAGAAGTGCTGTCAGGCCAATGCTCAGTTTGCAGAAATCTCCGA
GAACCTGAGAGCCAGAAGCAACAAAGATGCCAAGGATCCAACGACCAAGAACTCTCTGGAAACTCTGCTCTACAAGCCTGTGGACCGTGT
GACGAGGAGCACGCTGGTCCTCCATGACTTGCTGAAGCACACTCCTGCCAGCCACCCTGACCACCCCTTGCTGCAGGACGCCCTCCGCAT
CTCACAGAACTTCCTGTCCAGCATCAATGAGGAGATCACACCCCGACGGCAGTCCATGACGGTGAAGAAGGGAGAGCACCGGCAGCTGCT
GAAGGACAGCTTCATGGTGGAGCTGGTGGAGGGGGCCCGCAAGCTGCGCCACGTCTTCCTGTTCACCGACCTGCTTCTCTGCACCAAGCT
CAAGAAGCAGAGCGGAGGCAAAACGCAGCAGTATGACTGCAAATGGTACATTCCGCTCACGGATCTCAGCTTCCAGATGGTGGATGAACT
GGAGGCAGTGCCCAACATCCCCCTGGTGCCCGATGAGGAGCTGGACGCTTTGAAGATCAAGATCTCCCAGATCAAGAATGACATCCAGAG
AGAGAAGAGGGCGAACAAGGGCAGCAAGGCTACGGAGAGGCTGAAGAAGAAGCTGTCGGAGCAGGAGTCACTGCTGCTGCTTATGTCTCC
CAGCATGGCCTTCAGGGTGCACAGCCGCAACGGCAAGAGTTACACGTTCCTGATCTCCTCTGACTATGAGCGTGCAGAGTGGAGGGAGAA
CATCCGGGAGCAGCAGAAGAAGTGTTTCAGAAGCTTCTCCCTGACATCCGTGGAGCTGCAGATGCTGACCAACTCGTGTGTGAAACTCCA
GACTGTCCACAGCATTCCGCTGACCATCAATAAGGAAGATGATGAGTCTCCGGGGCTCTATGGGTTTCTGAATGTCATCGTCCACTCAGC
CACTGGATTTAAGCAGAGTTCAAATCTGTACTGCACCCTGGAGGTGGATTCCTTTGGGTATTTTGTGAATAAAGCAAAGACGCGCGTCTA
CAGGGACACAGCTGAGCCAAACTGGAACGAGGAATTTGAGATAGAGCTGGAGGGCTCCCAGACCCTGAGGATACTGTGCTATGAAAAGTG
TTACAACAAGACGAAGATCCCCAAGGAGGACGGCGAGAGCACGGACAGACTCATGGGGAAGGGCCAGGTCCAGCTGGACCCGCAGGCCCT
GCAGGACAGAGACTGGCAGCGCACCGTCATCGCCATGAATGGGATCGAAGTAAAGCTCTCGGTCAAGTTCAACAGCAGGGAGTTCAGCTT
GAAGAGGATGCCGTCCCGAAAACAGACAGGGGTCTTCGGAGTCAAGATTGCTGTGGTCACCAAGAGAGAGAGGTCCAAGGTGCCCTACAT
CGTGCGCCAGTGCGTGGAGGAGATCGAGCGCCGAGGCATGGAGGAGGTGGGCATCTACCGCGTGTCCGGTGTGGCCACGGACATCCAGGC
ACTGAAGGCAGCCTTCGACGTCAAAGCCCTTCAGCGGCCAGTAGCATCTGACTTTGAGCCTCAGGGTCTGAGTGAAGCCGCTCGTTGGAA
CTCCAAGGAAAACCTTCTCGCTGGACCCAGTGAAAATGACCCCAACCTTTTCGTTGCACTGTATGATTTTGTGGCCAGTGGAGATAACAC
TCTAAGCATAACTAAAGGTGAAAAGCTCCGGGTCTTAGGCTATAATCACAATGGGGAATGGTGTGAAGCCCAAACCAAAAATGGCCAAGG
CTGGGTCCCAAGCAACTACATCACGCCAGTCAACAGTCTGGAGAAACACTCCTGGTACCATGGGCCTGTGTCCCGCAATGCCGCTGAGTA
TCTGCTGAGCAGCGGGATCAATGGCAGCTTCTTGGTGCGTGAGAGTGAGAGCAGTCCTGGCCAGAGGTCCATCTCGCTGAGATACGAAGG
GAGGGTGTACCATTACAGGATCAACACTGCTTCTGATGGCAAGCTCTACGTCTCCTCCGAGAGCCGCTTCAACACCCTGGCCGAGTTGGT
TCATCATCATTCAACGGTGGCCGACGGGCTCATCACCACGCTCCATTATCCAGCCCCAAAGCGCAACAAGCCCACTGTCTATGGTGTGTC
CCCCAACTACGACAAGTGGGAGATGGAACGCACGGACATCACCATGAAGCACAAGCTGGGCGGGGGCCAGTACGGGGAGGTGTACGAGGG
CGTGTGGAAGAAATACAGCCTGACGGTGGCCGTGAAGACCTTGAAGGAGGACACCATGGAGGTGGAAGAGTTCTTGAAAGAAGCTGCAGT
CATGAAAGAGATCAAACACCCTAACCTGGTGCAGCTCCTTGGGGTCTGCACCCGGGAGCCCCCGTTCTATATCATCACTGAGTTCATGAC
CTACGGGAACCTCCTGGACTACCTGAGGGAGTGCAACCGGCAGGAGGTGAACGCCGTGGTGCTGCTGTACATGGCCACTCAGATCTCGTC
AGCCATGGAGTACCTGGAGAAGAAAAACTTCATCCACAGAGATCTTGCTGCCCGAAACTGCCTGGTAGGGGAGAACCACTTGGTGAAGGT
AGCTGATTTTGGCCTGAGCAGGTTGATGACAGGGGACACCTACACAGCCCATGCTGGAGCCAAGTTCCCCATCAAATGGACTGCACCCGA
GAGCCTGGCCTACAACAAGTTCTCCATCAAGTCCGACGTCTGGGCATTTGGAGTATTGCTTTGGGAAATTGCTACCTATGGCATGTCCCC
TTACCCGGGAATTGACCTGTCCCAGGTGTATGAGCTGCTAGAGAAGGACTACCGCATGGAGCGCCCAGAAGGCTGCCCAGAGAAGGTCTA
TGAACTCATGCGAGCATGTTGGCAGTGGAATCCCTCTGACCGGCCCTCCTTTGCTGAAATCCACCAAGCCTTTGAAACAATGTTCCAGGA
ATCCAGTATCTCAGACGAAGTGGAAAAGGAGCTGGGGAAACAAGGCGTCCGTGGGGCTGTGAGTACCTTGCTGCAGGCCCCAGAGCTGCC
CACCAAGACGAGGACCTCCAGGAGAGCTGCAGAGCACAGAGACACCACTGACGTGCCTGAGATGCCTCACTCCAAGGGCCAGGGAGAGAG
CGATCCTCTGGACCATGAGCCTGCCGTGTCTCCATTGCTCCCTCGAAAAGAGCGAGGTCCCCCGGAGGGCGGCCTGAATGAAGATGAGCG
CCTTCTCCCCAAAGACAAAAAGACCAACTTGTTCAGCGCCTTGATCAAGAAGAAGAAGAAGACAGCCCCAACCCCTCCCAAACGCAGCAG
CTCCTTCCGGGAGATGGACGGCCAGCCGGAGCGCAGAGGGGCCGGCGAGGAAGAGGGCCGAGACATCAGCAACGGGGCACTGGCTTTCAC
CCCCTTGGACACAGCTGACCCAGCCAAGTCCCCAAAGCCCAGCAATGGGGCTGGGGTCCCCAATGGAGCCCTCCGGGAGTCCGGGGGCTC
AGGCTTCCGGTCTCCCCACCTGTGGAAGAAGTCCAGCACGCTGACCAGCAGCCGCCTAGCCACCGGCGAGGAGGAGGGCGGTGGCAGCTC
CAGCAAGCGCTTCCTGCGCTCTTGCTCCGCCTCCTGCGTTCCCCATGGGGCCAAGGACACGGAGTGGAGGTCAGTCACGCTGCCTCGGGA
CTTGCAGTCCACGGGAAGACAGTTTGACTCGTCCACATTTGGAGGGCACAAAAGTGAGAAGCCGGCTCTGCCTCGGAAGAGGGCAGGGGA
GAACAGGTCTGACCAGGTGACCCGAGGCACAGTAACGCCTCCCCCCAGGCTGGTGAAAAAGAATGAGGAAGCTGCTGATGAGGTCTTCAA
AGACATCATGGAGTCCAGCCCGGGCTCCAGCCCGCCCAACCTGACTCCAAAACCCCTCCGGCGGCAGGTCACCGTGGCCCCTGCCTCGGG
CCTCCCCCACAAGGAAGAAGCTGGAAAGGGCAGTGCCTTAGGGACCCCTGCTGCAGCTGAGCCAGTGACCCCCACCAGCAAAGCAGGCTC
AGGTGCACCAGGGGGCACCAGCAAGGGCCCCGCCGAGGAGTCCAGAGTGAGGAGGCACAAGCACTCCTCTGAGTCGCCAGGGAGGGACAA
GGGGAAATTGTCCAGGCTCAAACCTGCCCCGCCGCCCCCACCAGCAGCCTCTGCAGGGAAGGCTGGAGGAAAGCCCTCGCAGAGCCCGAG
CCAGGAGGCGGCCGGGGAGGCAGTCCTGGGCGCAAAGACAAAAGCCACGAGTCTGGTTGATGCTGTGAACAGTGACGCTGCCAAGCCCAG
CCAGCCGGGAGAGGGCCTCAAAAAGCCCGTGCTCCCGGCCACTCCAAAGCCACAGTCCGCCAAGCCGTCGGGGACCCCCATCAGCCCAGC
CCCCGTTCCCTCCACGTTGCCATCAGCATCCTCGGCCCTGGCAGGGGACCAGCCGTCTTCCACCGCCTTCATCCCTCTCATATCAACCCG
AGTGTCTCTTCGGAAAACCCGCCAGCCTCCAGAGCGGATCGCCAGCGGCGCCATCACCAAGGGCGTGGTCCTGGACAGCACCGAGGCGCT
GTGCCTCGCCATCTCTAGGAACTCCGAGCAGATGGCCAGCCACAGCGCAGTGCTGGAGGCCGGCAAAAACCTCTACACGTTCTGCGTGAG
CTATGTGGATTCCATCCAGCAAATGAGGAACAAGTTTGCCTTCCGAGAGGCCATCAACAAACTGGAGAATAATCTCCGGGAGCTTCAGAT
CTGCCCGGCGACAGCAGGCAGTGGTCCAGCGGCCACTCAGGACTTCAGCAAGCTCCTCAGTTCGGTGAAGGAAATCAGTGACATAGTGCA
GAGGTAGCAGCAGTCAGGGGTCAGGTGTCAGGCCCGTCGGAGCTGCCTGCAGCACATGCGGGCTCGCCCATACCCGTGACAGTGGCTGAC
AAGGGACTAGTGAGTCAGCACCTTGGCCCAGGAGCTCTGCGCCAGGCAGAGCTGAGGGCCCTGTGGAGTCCAGCTCTACTACCTACGTTT
GCACCGCCTGCCCTCCCGCACCTTCCTCCTCCCCGCTCCGTCTCTGTCCTCGAATTTTATCTGTGGAGTTCCTGCTCCGTGGACTGCAGT
CGGCATGCCAGGACCCGCCAGCCCCGCTCCCACCTAGTGCCCCAGACTGAGCTCTCCAGGCCAGGTGGGAACGGCTGATGTGGACTGTCT
TTTTCATTTTTTTCTCTCTGGAGCCCCTCCTCCCCCGGCTGGGCCTCCTTCTTCCACTTCTCCAAGAATGGAAGCCTGAACTGAGGCCTT
GTGTGTCAGGCCCTCTGCCTGCACTCCCTGGCCTTGCCCGTCGTGTGCTGAAGACATGTTTCAAGAACCGCATTTCGGGAAGGGCATGCA
CGGGCATGCACACGGCTGGTCACTCTGCCCTCTGCTGCTGCCCGGGGTGGGGTGCACTCGCCATTTCCTCACGTGCAGGACAGCTCTTGA
TTTGGGTGGAAAACAGGGTGCTAAAGCCAACCAGCCTTTGGGTCCTGGGCAGGTGGGAGCTGAAAAGGATCGAGGCATGGGGCATGTCCT
TTCCATCTGTCCACATCCCCAGAGCCCAGCTCTTGCTCTCTTGTGACGTGCACTGTGAATCCTGGCAAGAAAGCTTGAGTCTCAAGGGTG
GCAGGTCACTGTCACTGCCGACATCCCTCCCCCAGCAGAATGGAGGCAGGGGACAAGGGAGGCAGTGGCTAGTGGGGTGAACAGCTGGTG
CCAAATAGCCCCAGACTGGGCCCAGGCAGGTCTGCAAGGGCCCAGAGTGAACCGTCCTTTCACACATCTGGGTGCCCTGAAAGGGCCCTT
CCCCTCCCCCACTCCTCTAAGACAAAGTAGATTCTTACAAGGCCCTTTCCTTTGGAACAAGACAGCCTTCACTTTTCTGAGTTCTTGAAG
CATTTCAAAGCCCTGCCTCTGTGTAGCCGCCCTGAGAGAGAATAGAGCTGCCACTGGGCACCTGCGCACAGGTGGGAGGAAAGGGCCTGG
CCAGTCCTGGTCCTGGCTGCACTCTTGAACTGGGCGAATGTCTTATTTAATTACCGTGAGTGACATAGCCTCATGTTCTGTGGGGGTCAT
CAGGGAGGGTTAGGAAAACCACAAACGGAGCCCCTGAAAGCCTCACGTATTTCACAGAGCACGCCTGCCATCTTCTCCCCGAGGCTGCCC
CAGGCCGGAGCCCAGATACGGGGGCTGTGACTCTGGGCAGGGACCCGGGGTCTCCTGGACCTTGACAGAGCAGCTAACTCCGAGAGCAGT
GGGCAGGTGGCCGCCCCTGAGGCTTCACGCCGGGAGAAGCCACCTTCCCACCCCTTCATACCGCCTCGTGCCAGCAGCCTCGCACAGGCC
CTAGCTTTACGCTCATCACCTAAACTTGTACTTTATTTTTCTGATAGAAATGGTTTCCTCTGGATCGTTTTATGCGGTTCTTACAGCACA
TCACCTCTTTGCCCCCGACGGCTGTGACGCAGCCGGAGGGAGGCACTAGTCACCGACAGCGGCCTTGAAGACAGAGCAAAGCGCCCACCC
AGGTCCCCCGACTGCCTGTCTCCATGAGGTACTGGTCCCTTCCTTTTGTTAACGTGATGTGCCACTATATTTTACACGTATCTCTTGGTA
TGCATCTTTTATAGACGCTCTTTTCTAAGTGGCGTGTGCATAGCGTCCTGCCCTGCCCCCTCGGGGGCCTGTGGTGGCTCCCCCTCTGCT
TCTCGGGGTCCAGTGCATTTTGTTTCTGTATATGATTCTCTGTGGTTTTTTTTGAATCCAAATCTGTCCTCTGTAGTATTTTTTAAATAA
ATCAGTGTTTACATTAGAA""".replace("\n", "").replace(" ", "")

bp_index = 4073
print(f"Sequence length: {len(full_sequence)} bp")
print(f"Breakpoint index: {bp_index}")


In [ ]:
results = []
KMER = 6  # Nucleotide Transformer uses 6-mer tokenization
MAX_TOKENS = tokenizer.model_max_length  # 2048 for this model

polomery = list(range(200, min(bp_index, len(full_sequence) - bp_index) + 1, 200))
print(f"Model max tokens: {MAX_TOKENS}")
print(f"Max bio_len for largest radius: {max(polomery)*2} nt = ~{max(polomery)*2//KMER+1} tokens (incl. CLS)")
print(f"Running {len(polomery)} radii...")

for polomer in tqdm(polomery, desc="Computing embeddings"):
    bio_segment = full_sequence[bp_index - polomer : bp_index + polomer]
    bio_len = len(bio_segment)

    # FIX: tokenize the biological sequence directly — no N-padding.
    # N characters are not in the 6-mer vocabulary and tokenize as single
    # characters, blowing past the 2048-token limit. Instead, let the
    # tokenizer pad with its own [PAD] token at the token level.
    inputs = tokenizer(
        bio_segment,
        return_tensors="pt",
        add_special_tokens=True,
        padding="max_length",
        max_length=MAX_TOKENS,
        truncation=False,   # fail loudly if a sequence is somehow too long
    ).to(device)

    # Breakpoint sits at position  (0-indexed) within bio_segment.
    # Token layout: [CLS] | bio_tokens | [PAD]...
    # So breakpoint token = 1 (CLS) + polomer // KMER
    token_idx = 1 + polomer // KMER

    # Attention mask is already correct from the tokenizer:
    # 1 for CLS + real bio tokens, 0 for [PAD] tokens.

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        bp_embedding = outputs.hidden_states[-1][0, token_idx, :].detach().cpu().float()

        results.append({
            "polomer":     polomer,
            "bio_seq_len": bio_len,
            "token_idx":   token_idx,
            "embedding":   bp_embedding
        })

    del outputs, inputs
    torch.cuda.empty_cache()

print("Done.")


In [ ]:
if not results:
    print("Error: results list is empty.")
else:
    # Reference embedding = largest radius (last entry)
    ref_emb = results[-1]["embedding"]
    summary = []

    for item in results:
        cos_sim = F.cosine_similarity(
            item["embedding"].unsqueeze(0),
            ref_emb.unsqueeze(0)
        ).item()
        summary.append({
            "Polomer_bp":        item["polomer"],
            "Biologicka_Delka":  item["bio_seq_len"],
            "Token_idx":         item["token_idx"],
            "Cosine_Similarity": round(cos_sim, 5)
        })

    df_results = pd.DataFrame(summary)
    print(df_results.to_string(index=False))
    df_results.to_csv("analyza_final_v3.csv", index=False)
    print("\nSaved to analyza_final_v3.csv")
